In [1]:
# Track 2 Model B (UNETR) evaluation: stage data + run inference on val (63) and test (56) cases.
import os, glob, json, subprocess, sys
try:
    import monai
    print('monai', monai.__version__)
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai>=1.5'], check=True)
    import monai
    print('monai installed', monai.__version__)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())

os.chdir('/kaggle/working')
if not os.path.isdir('Neuroshield'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pankurajasaivardhan/Neuroshield.git'], check=True)
sys.path.insert(0, '/kaggle/working/Neuroshield')
from track2_brain_tumor.src.train_model_b_unetr import case_dicts, transforms_val, remap_label
from monai.networks.nets import UNETR
from monai.inferers import sliding_window_inference
from monai.data import Dataset, DataLoader, decollate_batch
from monai.transforms import Invertd, AsDiscreted, Compose
import pandas as pd
import numpy as np

# --- manifest: same fold0 split as training (249/63/56) ---
DS = '/kaggle/input/datasets/rajasaivardhan/neuroshield-track2-model-a'
splits = json.load(open(sorted(glob.glob(os.path.join(DS, '**/nnUNet_preprocessed/**/splits_final.json'), recursive=True) or glob.glob(os.path.join(DS, '**/splits_final.json'), recursive=True))[0]))
fold = splits[0]
test_ids = sorted({os.path.basename(p).replace('.nii.gz', '').replace('.nii', '')
                   for p in glob.glob(os.path.join(DS, '**/test_predictions/**/*.nii*'), recursive=True)})
print('fold0 train/val:', len(fold['train']), len(fold['val']), '| test:', len(test_ids))
assert len(fold['train']) == 249 and len(fold['val']) == 63 and len(test_ids) == 56, 'split mismatch'
rows = [(c, 'train') for c in fold['train']] + [(c, 'val') for c in fold['val']] + [(c, 'test') for c in test_ids]
pd.DataFrame(rows, columns=['case_id', 'split']).to_csv('/kaggle/working/manifest.csv', index=False)

# --- Model A test preds staging (fast-fail before heavy staging) ---
A_TEST = sorted(glob.glob(os.path.join(DS, '**/test_predictions/*.nii'), recursive=True))
found = {}
for p in A_TEST:
    cid = os.path.basename(p).replace('.nii.gz', '').replace('.nii', '')
    found[cid] = p
missing_a = [c for c in test_ids if c not in found]
print('Model A test preds cover test ids:', len(test_ids) - len(missing_a), '/', len(test_ids), '| missing:', missing_a)
assert not missing_a, 'Model A test predictions missing for some test cases'
a_out = '/kaggle/working/model_a_test_preds'
os.makedirs(a_out, exist_ok=True)
for c in test_ids:
    dst = os.path.join(a_out, c + '.nii')
    if not os.path.exists(dst):
        os.symlink(os.path.realpath(found[c]), dst)
print('staged Model A test preds:', len(os.listdir(a_out)))

# --- stage raw val + test cases with plain .nii names (MONAI/nibabel need real names) ---
GT = glob.glob('/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
stage = '/kaggle/working/brats'
MODS = ['flair', 't1', 't1ce', 't2']
n = 0
for cid in fold['val'] + test_ids:
    cdir = os.path.join(GT, cid)
    if not os.path.isdir(cdir):
        print('MISSING', cid)
        continue
    out = os.path.join(stage, cid)
    os.makedirs(out, exist_ok=True)
    for m in MODS + ['seg']:
        src = os.path.join(cdir, f'{cid}_{m}.nii')
        dst = os.path.join(out, f'{cid}_{m}.nii')
        if not os.path.exists(dst):
            os.symlink(os.path.realpath(src), dst)
    n += 1
print('staged cases:', n)
assert n == 119, f'expected 119 staged cases, got {n}'

# --- UNETR inference on val + test, predictions restored to original image space ---
ckpt = glob.glob('/kaggle/input/**/unetr_best.pt', recursive=True)
print('ckpt candidates:', ckpt)
assert ckpt, 'unetr_best.pt not found in inputs'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNETR(in_channels=4, out_channels=4, img_size=(96, 96, 96), feature_size=16, hidden_size=768, mlp_dim=3072, num_heads=12, norm_name='instance', res_block=True, dropout_rate=0.1).to(device)
model.load_state_dict(torch.load(ckpt[0], map_location=device))
model.eval()
print('checkpoint loaded')

val_tf = transforms_val()
post = Compose([
    Invertd(keys='pred', transform=val_tf, orig_keys='image', meta_keys='pred_meta_dict', orig_meta_keys='image_meta_dict', nearest_interp=True, to_tensor=True),
    AsDiscreted(keys='pred', argmax=True),
])
import nibabel as nib
for split in ('val', 'test'):
    out_dir = f'/kaggle/working/outputs/unetr_{split}_predictions'
    os.makedirs(out_dir, exist_ok=True)
    ds = Dataset(case_dicts('/kaggle/working/manifest.csv', stage, split), transform=val_tf)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2)
    done = 0
    for batch in loader:
        cid = batch['case_id'][0]
        out_path = os.path.join(out_dir, f'{cid}.nii.gz')
        if os.path.exists(out_path):
            done += 1
            continue
        x = batch['image'].to(device)
        with torch.no_grad():
            batch['pred'] = sliding_window_inference(x, (96, 96, 96), 4, model, overlap=0.5)
        inv = [post(i) for i in decollate_batch(batch)][0]
        pred = inv['pred'][0].cpu().numpy().astype(np.uint8)
        affine = inv['pred'].meta['affine']
        if hasattr(affine, 'numpy'):
            affine = affine.cpu().numpy()
        nib.save(nib.Nifti1Image(pred, np.asarray(affine)), out_path)
        done += 1
        if done % 10 == 0 or done == len(ds):
            print(f'{split}: {done}/{len(ds)} predictions written', flush=True)
    print(f'{split} inference complete:', done)
print('INFERENCE DONE')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.0 MB/s eta 0:00:00
monai installed 1.6.1
torch 2.11.0+cu128 | cuda: True | gpus: 2


Cloning into 'Neuroshield'...


fold0 train/val: 249 63 | test: 56
Model A test preds cover test ids: 56 / 56 | missing: []
staged Model A test preds: 56
staged cases: 119
ckpt candidates: ['/kaggle/input/notebooks/rajasaivardhan/track2-model-b-unetr/outputs/unetr_best.pt']


nvImageCodec is not available; skipping pydicom decoder plugin registration.


checkpoint loaded


/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


val: 10/63 predictions written
val: 20/63 predictions written
val: 30/63 predictions written
val: 40/63 predictions written
val: 50/63 predictions written
val: 60/63 predictions written
val: 63/63 predictions written
val inference complete: 63
test: 10/56 predictions written
test: 20/56 predictions written
test: 30/56 predictions written
test: 40/56 predictions written
test: 50/56 predictions written
test: 56/56 predictions written
test inference complete: 56
INFERENCE DONE


In [2]:
# Track 2 Model B evaluation: held-out test + val region metrics vs GT, then test reliability/disagreement (Model A vs UNETR).
import os, glob, json, subprocess, sys
import numpy as np
import pandas as pd
import nibabel as nib
sys.path.insert(0, '/kaggle/working/Neuroshield')
from track2_brain_tumor.src.eval_model_a_test import REGIONS, dice_iou, hd95

GT = glob.glob('/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
manifest = pd.read_csv('/kaggle/working/manifest.csv')

def region_metrics(split):
    rows = []
    for cid in manifest[manifest['split'] == split]['case_id']:
        pred_path = f'/kaggle/working/outputs/unetr_{split}_predictions/{cid}.nii.gz'
        gt_path = os.path.join(GT, cid, f'{cid}_seg.nii')
        pred = nib.load(pred_path).get_fdata().astype(int)
        gt_img = nib.load(gt_path)
        gt = np.where(gt_img.get_fdata().astype(int) == 4, 3, gt_img.get_fdata().astype(int))
        spacing = gt_img.header.get_zooms()[:3]
        row = {'case_id': cid}
        for (region, fn) in REGIONS.items():
            (p_m, g_m) = (fn(pred), fn(gt))
            (d, i) = dice_iou(p_m, g_m)
            row[f'{region}_dice'] = d
            row[f'{region}_iou'] = i
            row[f'{region}_hd95_mm'] = hd95(p_m, g_m, spacing)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(f'/kaggle/working/outputs/unetr_{split}_per_case.csv', index=False)
    agg = df.drop(columns=['case_id']).mean(numeric_only=True).to_dict()
    json.dump({'n_cases': len(rows), 'model': 'UNETR (Model B)', 'split': split,
               'aggregate_mean': agg,
               'note': 'UNETR trained 40 epochs (best val ckpt), sliding-window inference overlap 0.5, predictions inverted to original space.'},
              open(f'/kaggle/working/outputs/unetr_{split}_summary.json', 'w'), indent=2)
    print(f'[{split}] n={len(rows)} ' + json.dumps({k: round(v, 4) for k, v in agg.items()}), flush=True)
    return df

test_df = region_metrics('test')
val_df = region_metrics('val')

# --- test reliability/disagreement: Model A test preds vs UNETR test preds vs GT ---
# (Model A val predictions do not exist; eval_reliability_seg fits no thresholds,
#  so running it on the 56-case held-out test set is rules-clean.)
def real_gz(src, dst):
    img = nib.load(src)
    nib.save(nib.Nifti1Image(np.asanyarray(img.dataobj).astype(np.uint8), img.affine), dst)

a_dir = '/kaggle/working/rel_model_a_test'
g_dir = '/kaggle/working/rel_test_gt'
b_dir = '/kaggle/working/outputs/unetr_test_predictions'
for d in (a_dir, g_dir):
    os.makedirs(d, exist_ok=True)
test_cids = manifest[manifest['split'] == 'test']['case_id'].tolist()
missing = []
for cid in test_cids:
    a_src = f'/kaggle/working/model_a_test_preds/{cid}.nii'
    g_src = os.path.join(GT, cid, f'{cid}_seg.nii')
    if not (os.path.exists(a_src) and os.path.exists(g_src)):
        missing.append(cid)
        continue
    a_dst = os.path.join(a_dir, f'{cid}.nii.gz')
    g_dst = os.path.join(g_dir, f'{cid}.nii.gz')
    if not os.path.exists(a_dst):
        real_gz(a_src, a_dst)
    if not os.path.exists(g_dst):
        real_gz(g_src, g_dst)
print('staged test A preds + GT (real gzip):', len(test_cids) - len(missing), '| missing:', missing)
assert not missing, 'test reliability staging incomplete'
a_names = {os.path.basename(p) for p in glob.glob(os.path.join(a_dir, '*.nii.gz'))}
b_names = {os.path.basename(p) for p in glob.glob(os.path.join(b_dir, '*.nii.gz'))}
print('name overlap A/B:', len(a_names & b_names))
assert len(a_names & b_names) == 56, 'test pred name mismatch between Model A and UNETR'

os.chdir('/kaggle/working/Neuroshield')
r = subprocess.run([sys.executable, '-m', 'track2_brain_tumor.src.eval_reliability_seg',
                    '--pred-a', a_dir, '--pred-b', b_dir, '--gt-dir', g_dir,
                    '--out', '/kaggle/working/outputs'], check=False)
print('eval_reliability_seg exit', r.returncode)
assert r.returncode == 0, 'seg reliability eval failed - see log'
rel = json.load(open('/kaggle/working/outputs/track2_reliability.json'))
cases = rel['per_case']
dr = [c['disagreement']['foreground_disagreement_rate'] for c in cases if 'disagreement' in c]
iou1 = [c['disagreement']['per_class_iou'].get('1') for c in cases if 'disagreement' in c]
iou2 = [c['disagreement']['per_class_iou'].get('2') for c in cases if 'disagreement' in c]
iou3 = [c['disagreement']['per_class_iou'].get('3') for c in cases if 'disagreement' in c]
def m(x):
    x = [v for v in x if v is not None]
    return float(np.mean(x)) if x else None
summary = {'n_cases': len(cases), 'split': 'test', 'with_disagreement': len(dr),
           'mean_foreground_disagreement_rate': m(dr),
           'mean_per_class_iou': {'1_necrotic_core': m(iou1), '2_edema': m(iou2), '3_enhancing': m(iou3)}}
json.dump(summary, open('/kaggle/working/outputs/track2_reliability_summary.json', 'w'), indent=2)
print('RELIABILITY SUMMARY ' + json.dumps(summary), flush=True)

# copy Model A test per-case CSV if attached (for paired figures)
for p in glob.glob('/kaggle/input/**/model_a_test_per_case.csv', recursive=True):
    subprocess.run(['cp', p, '/kaggle/working/outputs/model_a_test_per_case.csv'])
    print('copied Model A test per-case CSV from', p)
print('EVAL COMPLETE')


[test] n=56 {"WT_dice": 0.7913, "WT_iou": 0.6921, "WT_hd95_mm": 8.8574, "TC_dice": 0.6279, "TC_iou": 0.5181, "TC_hd95_mm": 13.3171, "ET_dice": 0.6358, "ET_iou": 0.5223, "ET_hd95_mm": 12.725}
[val] n=63 {"WT_dice": 0.8188, "WT_iou": 0.7281, "WT_hd95_mm": 4.9492, "TC_dice": 0.7104, "TC_iou": 0.6033, "TC_hd95_mm": 7.0084, "ET_dice": 0.6803, "ET_iou": 0.5695, "ET_hd95_mm": 7.0199}
staged test A preds + GT (real gzip): 56 | missing: []
name overlap A/B: 56
Wrote 56 case records. Voxel calibration requires saved probability maps (--save-probabilities on predict).
eval_reliability_seg exit 0
RELIABILITY SUMMARY {"n_cases": 56, "split": "test", "with_disagreement": 56, "mean_foreground_disagreement_rate": 0.3687533021446449, "mean_per_class_iou": {"1_necrotic_core": 0.35779744683419273, "2_edema": 0.5937910158580867, "3_enhancing": 0.5749730450865619}}
EVAL COMPLETE
